In [1]:
import torch
from torch_geometric.data import Data
from torch_geometric.datasets import Reddit
from torch_geometric.transforms import NormalizeFeatures

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
# Mengunduh dataset Reddit Binary
dataset = Reddit(root='data/Reddit', transform=NormalizeFeatures())

# Mengambil data pertama dari dataset
data = dataset[0]

In [4]:
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.num_edges}")
print(f"Number of node features: {data.num_node_features}")
print(f"Number of classes: {dataset.num_classes}")

Number of nodes: 232965
Number of edges: 114615892
Number of node features: 602
Number of classes: 41


In [5]:
# Mengambil node features (x) dan edge index
x = data.x
edge_index = data.edge_index

# Membuat edge_attr (biasanya berupa jarak atau bobot edge)
# Karena dataset Reddit Binary tidak menyediakan edge_attr, kita bisa membuatnya secara manual
edge_attr = torch.ones(edge_index.size(1), 1)  # Misalnya, semua edge memiliki bobot 1

# Membuat batch (setiap node dalam grafik memiliki batch index yang sama)
batch = torch.zeros(data.num_nodes, dtype=torch.long)

# Membuat tensor target (y)
y = data.y

In [6]:
# Membuat objek Data
graph_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, batch=batch)

In [7]:
# Menambahkan posisi acak ke dataset
graph_data.pos = torch.rand(graph_data.num_nodes, 3)  # Posisi acak dalam ruang 3D

# Menambahkan batch (misalnya, semua node dalam satu batch)
graph_data.batch = torch.zeros(graph_data.num_nodes, dtype=torch.long)

graph_data.atom_type = torch.randint(0, 200, (graph_data.num_nodes,))

In [8]:
graph_data = graph_data.to(device)

In [9]:
from torch_geometric.loader import DataLoader

# Membuat DataLoader
loader = DataLoader([graph_data], batch_size=1, shuffle=True)

# Iterasi melalui DataLoader
for batch in loader:
    print(batch)

DataBatch(x=[232965, 602], edge_index=[2, 114615892], edge_attr=[114615892, 1], y=[232965], batch=[232965], pos=[232965, 3], atom_type=[232965], ptr=[2])


In [10]:
from models.drgin5 import DRGIN5

In [11]:
# Inisialisasi model DRGIN
model = DRGIN5(
    node_dimses=[[64, 64], [64, 64]],  # Contoh konfigurasi node_dimses
    edge_dimses=[[1, 64], [64, 64]],   # Contoh konfigurasi edge_dimses
    dropout_rate=0.5,
    cutoff=10.0,
    max_neighbors=32,
    aggr='sum'
)
model = model.to(device)

In [12]:
# Forward pass
output = model(graph_data)
print(output)

edge_attr sebelum MLP: torch.Size([7454880, 1])
edge_attr setelah MLP: torch.Size([7454880, 64])


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.78 GiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 9.79 GiB is allocated by PyTorch, and 146.17 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from sklearn.metrics import accuracy_score

# Prediksi kelas
pred = output.argmax(dim=1)

# Menghitung accuracy
accuracy = accuracy_score(y.numpy(), pred.numpy())
print(f"Accuracy: {accuracy}")

NameError: name 'output' is not defined

In [ ]:
torch.cuda.reset_peak_memory_stats()